# Vehicle Failure Survival Analysis Walkthrough

This notebook uses the repository's synthetic commercial-vehicle component histories. Run `python scripts/run_all.py` first so the model artifacts and reports exist.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.duration.survfunc import SurvfuncRight, survdiff

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "data/synthetic/fleet_survival.csv", parse_dates=["cohort_start_date"])
df.head()

## Censoring

`event=1` indicates an observed failure. `event=0` is right censoring, meaning the component was still operating when observation ended.

In [ ]:
summary = {
    "rows": len(df),
    "event_rate": df["event"].mean(),
    "censoring_rate": 1 - df["event"].mean(),
    "median_observed_months": df["duration_months"].median(),
}
summary

## Kaplan-Meier survival curves by operating load

In [ ]:
threshold = df["load_factor"].median()
group = (df["load_factor"] >= threshold).astype(int)
fig, ax = plt.subplots(figsize=(8,5))
for label, value in [("Lower load", 0), ("Higher load", 1)]:
    subset = df[group == value]
    km = SurvfuncRight(subset["duration_months"], subset["event"])
    ax.step(km.surv_times, km.surv_prob, where="post", label=label)
ax.set_xlabel("Component age (months)")
ax.set_ylabel("Estimated survival probability")
ax.legend();

stat, p = survdiff(df["duration_months"].to_numpy(), df["event"].to_numpy(), group.to_numpy())
print("log-rank statistic:", stat, "p-value:", p)

## Model comparison

The training pipeline fits both Cox PH and Weibull AFT, then compares discrimination and censored probability error.

In [ ]:
metrics = json.loads((ROOT / "artifacts/metrics.json").read_text())
pd.DataFrame({
    "model": ["Cox PH", "Weibull AFT"],
    "c_index": [metrics["cox_ph"]["c_index"], metrics["weibull_aft"]["c_index"]],
    "brier_average": [metrics["cox_ph"]["integrated_brier_proxy"], metrics["weibull_aft"]["integrated_brier_proxy"]],
})

## Cox hazard ratios

Numeric features are standardized before fitting, so each numeric hazard ratio corresponds to approximately one standard deviation of movement in that feature.

In [ ]:
hr = pd.read_csv(ROOT / "reports/tables/cox_hazard_ratios.csv")
hr

## Calibration

Predicted risk is grouped into quantiles and compared with Kaplan-Meier observed event risk.

In [ ]:
pd.read_csv(ROOT / "reports/tables/cox_calibration_24m.csv")

## Proportional-hazards diagnostic

The exported table checks simple correlations between Schoenfeld residuals and log event time. It is a diagnostic heuristic, not a full formal PH test.

In [ ]:
pd.read_csv(ROOT / "reports/tables/cox_ph_diagnostics.csv")

## Important portfolio limitation

The dataset is synthetic. The project demonstrates statistical reasoning, right-censoring methodology, evaluation and production engineering. The reported performance must not be presented as real OEM field performance.